# EC interpolation to non-zero $\phi$ with stats
This notebook tests if EC can accurately interpolate within the training region. Everything is analytic in the 
good region of $\phi$, which is where we train EC, so EC should have no problem predicting energy and $(r_\theta^2)$ in this
region. This notebook is here as a sanity check to test this.

In [1]:
from src.ecensemble import ECEnsemble
from src.ec import ECSystem, TakagiRegularization
from src.dvr import DVR
from src.utils import get_plots_path

import numpy as np
from numpy import pi, exp
import random
import matplotlib.pyplot as plt

import logging
logging.basicConfig(level=logging.WARNING, format="[%(levelname)s] %(name)s: %(message)s")
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [2]:
save_plots = True
plots_folder = get_plots_path()/ "ec_interpolation_wstats"
plots_folder.mkdir(exist_ok=True, parents=True)

In [3]:
seed = 1

random.seed(seed)
np.random.seed(seed)

In [4]:
# define system parameters
n = 256                      # number of DVR grid points
n_ecsystems = 128            # number of EC computations per phi
k = 1                        # number of states to sample per phi
phi = pi / 24                # rotation angle
real_res = 1.606 - 1j*0.047  # expected value of resonance
L = 50.0                     # system size

def gauss_s(r):
    return 2.0 * exp(-((r-3.0)/1.5)**2) 
#gauss_s = lambda r: 2.0 * exp(-((r-3.0)/1.5)**2) # potential

### Compute exact and training data for resonance energies and rotated $(r^2)$.

In [5]:
# compute exact data
phi_predict = np.linspace(pi/24, pi/6, 5)

dvr_system = DVR(n, L, rotation_angle=phi, potential=gauss_s)
exact_energies, exact_rrs = [], []
for phi in phi_predict:
    dvr_system.reset_rotation_angle(phi)
    eigval, eigstate = dvr_system.closest_to_resonance(real_res)
    rr = dvr_system.compute_rr(eigstate[:, 0], rotate_rr=True)
    exact_energies.append(eigval[0])
    exact_rrs.append(rr)

# compute training data
phi_training_range, num_phi_points = [pi/24, pi/6], 20

train_energies, train_rrs = [], []
for phi in np.linspace(phi_training_range[0], phi_training_range[1], 20):
    dvr_system.reset_rotation_angle(phi)
    eigval, eigstate = dvr_system.closest_to_resonance(real_res)
    rr = dvr_system.compute_rr(eigstate[:, 0], rotate_rr=True)
    train_energies.append(eigval[0])
    train_rrs.append(rr)

### Train multiple EC systems, predicting at $\phi=\pi/24$ (in the training region) and compute stats.

In [6]:
# construct EC ensemble and train all ECs
EC_ensemble = ECEnsemble(n_ecsystems, dvr_system, real_res)
EC_ensemble.train(phi_training_range, num_phi_points, k)
EC_ensemble.construct_bases()

# predict energies, <r^2> and <r^2> IQR
predict_energies_stats, predict_rrs_stats = EC_ensemble.predict_energies_rrs_wstats(phi_predict=pi/24, rotate_rr=True)

AttributeError: 'ECEnsemble' object has no attribute 'predict_energies_rrs_wstats'

In [ ]:
energy_median, energy_err68, energy_err95 = predict_energies_stats
rr_median, rr_err68, rr_err95 = predict_rrs_stats

In [ ]:
def plot_wstats(fig, ax, median, err68, err95):
    err68, err95 = err68[:, np.newaxis], err95[:, np.newaxis]
    ax.scatter(np.real(median), np.imag(median), c='red', marker='x', label="Predict (median)")
    ax.errorbar(np.real(median), np.imag(median),
            yerr=np.imag(err68),
            xerr=np.real(err68),
            fmt='none', ecolor='red', elinewidth=2.0, alpha=1.0, label='Predict (68.2% int)')
    ax.errorbar(np.real(median), np.imag(median),
            yerr=np.imag(err95),
            xerr=np.real(err95),
            fmt='none', ecolor='red', elinewidth=2.0, alpha=.4, label='Predict (95.4% int)')

# plot results with error bands
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(np.real(train_rrs), np.imag(train_rrs), alpha=0.6, marker='x', label="Training")
#ax.set_aspect('equal')
ax.scatter(np.real(exact_rrs), np.imag(exact_rrs), marker='x', label="Exact")
plot_wstats(fig, ax, rr_median, rr_err68, rr_err95)
ax.set_xlabel(r"$Re(\langle r^2 \rangle)$")
ax.set_ylabel(r"$Im(\langle r^2 \rangle)$")
ax.set_title(r"$\langle r^2\rangle$ for $\phi\in [\pi/24,\pi/6]$")
ax.legend()
if save_plots:
    plt.savefig(plots_folder / "ec_radius_stats.png")
plt.show()

fig, ax = plt.subplots()
#ax.set_aspect('equal')
ax.scatter(np.real(train_energies), np.imag(train_energies), marker='x', alpha=0.6, label=r"Training")
ax.scatter(np.real(exact_energies), np.imag(exact_energies), marker='x', label="Exact")
plot_wstats(fig, ax, energy_median, energy_err68, energy_err95)
ax.set_xlabel(r"$Re(E)$")
ax.set_ylabel(r"$Im(E)$")
ax.set_title(r"$E$ for $\phi\in [\pi/24,\pi/6]$")
ax.legend()
if save_plots:
    plt.savefig(plots_folder / "ec_energy_stats.png")
plt.show()

In [ ]:
train_energy_median, train_energy_err68, train_energy_err95 = ECEnsemble.compute_stats(np.array(train_energies))
train_rrs_median, train_rrs_err68, train_rrs_err95 = ECEnsemble.compute_stats(np.array(train_rrs))

print(f"Training energies: ({np.real(train_energy_median):.6f} +- {np.max(np.abs(np.real(train_energy_err95))):.6f})"
      f" + i({np.imag(train_energy_median):.6f} +- {np.max(np.abs(np.imag(train_energy_err95))):.6f})")
print(f"Training rrs: ({np.real(train_rrs_median):.6f} +- {np.max(np.abs(np.real(train_rrs_err95))):.6f})"
      f" + i({np.imag(train_rrs_median):.6f} +- {np.max(np.abs(np.imag(train_rrs_err95))):.6f})")


print(
    f"EC predicted <r^2> (in training region) = "
    f"({np.real(rr_median):.6f} ± {np.max(np.abs(np.real(rr_err95))):.6f}) "
    f"+ i({np.imag(rr_median):.6f} ± {np.max(np.abs(np.imag(rr_err95))):.6f})"
)

print(
    f"EC predicted E (in training region) = "
    f"({np.real(energy_median):.6f} ± {np.max(np.abs(np.real(energy_err95))):.6f}) "
    f"+ i({np.imag(energy_median):.6f} ± {np.max(np.abs(np.imag(energy_err95))):.6f})"
)

## Conclusion:
EC can accurately interpolate in the training region, as it should be able to.